# NCCT Stroke Segmentation with MMSegmentation

Segment ischemic stroke regions from Non-Contrast CT (NCCT) brain images using the [MMSegmentation](https://github.com/open-mmlab/mmsegmentation) framework.

**This notebook:**
1. Clones the [ncct-segmentation](https://github.com/lhfazry/ncct-segmentation) repo (MMSegmentation fork)
2. Installs MMCV, MMEngine, and the project
3. Downloads the NCCT stroke dataset
4. Trains a U-Net model
5. Evaluates and visualizes results

---
**Runtime**: T4 GPU or better recommended  
**Storage needed**: ~10GB

## 1. Environment Setup

### Verify GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### Install Dependencies

MMSegmentation requires:
- **MMCV**: Install via `mim` (the recommended way)
- **MMEngine**: Install via pip
- **Project-specific**: Install the forked repo in editable mode

In [ ]:
# Install MIM (MMCV Installer) and MMCV
!pip install -q openmim
!mim install mmcv>=2.0.0 -q

# Install MMEngine
!pip install -q mmengine

In [ ]:
import os
from pathlib import Path

# Clone the repository
REPO_URL = "https://github.com/lhfazry/ncct-segmentation"
REPO_DIR = "/content/ncct-segmentation"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git status --short

In [ ]:
# Install the project in editable mode
# This registers mmseg as a package and makes tools/*.py runnable
!pip install -e . -q
!pip install albumentations -q

import mmseg
from mmseg.utils import register_all_modules
register_all_modules()
print(f"mmseg version: {mmseg.__version__}")

---
## 2. Dataset Preparation

Download the NCCT brain dataset, extract it, and verify the directory structure.

In [ ]:
import gdown
import zipfile
import shutil

# Google Drive file ID (same dataset as original notebook)
FILE_ID = "1o0b6Nqs89zYoyRcnih5oGS7sk0bIrImo"
ZIP_PATH = "/content/dataset.zip"
EXTRACT_DIR = "/content/dataset_raw"

if not os.path.exists(ZIP_PATH):
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    print("Downloading dataset from Google Drive...")
    gdown.download(url, ZIP_PATH, quiet=False)
else:
    print("Dataset zip already downloaded.")

In [ ]:
# Extract dataset
if not os.path.exists(EXTRACT_DIR):
    print("Extracting dataset...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

In [ ]:
# Organize into MMSegmentation-compatible structure
# Expected structure:
# data/ncct/
#   train/images/  -> *.png
#   train/masks/   -> *.png
#   val/images/    -> *.png
#   val/masks/     -> *.png
#   test/images/   -> *.png
#   test/masks/    -> *.png

DATA_ROOT = "/content/data/ncct"
os.makedirs(DATA_ROOT, exist_ok=True)

# Check what's inside the extracted zip
extracted_contents = os.listdir(EXTRACT_DIR)
print(f"Extracted contents: {extracted_contents}")

# The dataset likely has train/val/test dirs with images/ and masks/ inside
# If already structured correctly, just symlink
expected_split_dirs = ["train", "val", "test"]
found_splits = [d for d in expected_split_dirs if os.path.isdir(os.path.join(EXTRACT_DIR, d))]

if found_splits:
    print(f"Found splits: {found_splits}")
    # If already in correct format, just copy
    for split in found_splits:
        src_img = os.path.join(EXTRACT_DIR, split, "images")
        src_mask = os.path.join(EXTRACT_DIR, split, "masks")
        if os.path.isdir(src_img) and os.path.isdir(src_mask):
            dst_img = os.path.join(DATA_ROOT, split, "images")
            dst_mask = os.path.join(DATA_ROOT, split, "masks")
            os.makedirs(os.path.dirname(dst_img), exist_ok=True)
            if not os.path.exists(dst_img):
                shutil.copytree(src_img, dst_img)
            if not os.path.exists(dst_mask):
                shutil.copytree(src_mask, dst_mask)
else:
    # Try to auto-detect the structure
    for root, dirs, files in os.walk(EXTRACT_DIR):
        for d in dirs:
            if d in ("images", "masks"):
                parent = os.path.basename(os.path.dirname(os.path.join(root, d)))
                print(f"  Found '{d}' under directory '{parent}'")

print(f"\nDataset organized at: {DATA_ROOT}")

In [ ]:
# Verify dataset structure
for split in ["train", "val", "test"]:
    img_dir = os.path.join(DATA_ROOT, split, "images")
    mask_dir = os.path.join(DATA_ROOT, split, "masks")
    if os.path.isdir(img_dir) and os.path.isdir(mask_dir):
        images = sorted(os.listdir(img_dir))
        masks = sorted(os.listdir(mask_dir))
        print(f"{split:5s}: {len(images):4d} images, {len(masks):4d} masks")
        if images and masks:
            print(f"         Sample: {images[0]}")
    else:
        print(f"{split:5s}: NOT FOUND")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Quick visualization of dataset samples
img_dir = os.path.join(DATA_ROOT, "train", "images")
mask_dir = os.path.join(DATA_ROOT, "train", "masks")

if os.path.isdir(img_dir):
    imgs = sorted(os.listdir(img_dir))[:3]
    fig, axes = plt.subplots(len(imgs), 2, figsize=(8, 3*len(imgs)))
    for i, fname in enumerate(imgs):
        img = Image.open(os.path.join(img_dir, fname)).convert("L")
        mask = Image.open(os.path.join(mask_dir, fname)).convert("L")

        axes[i][0].imshow(img, cmap="gray")
        axes[i][0].set_title(f"Input: {fname}")
        axes[i][0].axis("off")

        axes[i][1].imshow(mask, cmap="gray")
        axes[i][1].set_title(f"Mask (stroke in white)")
        axes[i][1].axis("off")

    plt.tight_layout()
    plt.show()
    
    # Show class distribution
    total_pixels = 0
    stroke_pixels = 0
    for fname in os.listdir(mask_dir)[:100]:
        m = np.array(Image.open(os.path.join(mask_dir, fname)).convert("L"))
        total_pixels += m.size
        stroke_pixels += (m > 127).sum()
    print(f"Class balance (first 100 training samples):")
    print(f"  Background: {(total_pixels - stroke_pixels) / total_pixels * 100:.2f}%")
    print(f"  Stroke:     {stroke_pixels / total_pixels * 100:.2f}%")

---
## 3. Configuration

The training config is at `configs/ncct/unet_stroke_ncct.py`.  
It inherits from:
- `_base_/models/fcn_unet_s5-d16.py` — U-Net architecture
- `_base_/default_runtime.py` — Logging, visualization, checkpoint settings
- `_base_/schedules/schedule_20k.py` — SGD optimizer, PolyLR scheduler, 20K iterations

We update `data_root` in the config to point to our Colab path.

In [ ]:
from mmengine.config import Config

CFG_PATH = "/content/ncct-segmentation/configs/ncct/unet_stroke_ncct.py"
cfg = Config.fromfile(CFG_PATH)
print(cfg.pretty_text)

In [ ]:
# Update config for Colab paths
cfg.data_root = "/content/data/ncct/"

# Update all dataloader data_roots
cfg.train_dataloader.dataset.data_root = cfg.data_root
cfg.val_dataloader.dataset.data_root = cfg.data_root
cfg.test_dataloader.dataset.data_root = cfg.data_root

# Set work directory
WORK_DIR = "/content/work_dirs/unet_stroke_ncct"
cfg.work_dir = WORK_DIR
os.makedirs(WORK_DIR, exist_ok=True)

# Use mixed precision for faster training
cfg.optim_wrapper.type = "AmpOptimWrapper"
cfg.optim_wrapper.loss_scale = "dynamic"

# Save updated config for reference
UPDATED_CFG_PATH = "/content/unet_stroke_ncct_colab.py"
cfg.dump(UPDATED_CFG_PATH)
print(f"Config saved to {UPDATED_CFG_PATH}")
print(f"data_root: {cfg.data_root}")
print(f"work_dir:  {cfg.work_dir}")

---
## 4. Training

Run the MMSegmentation training pipeline. 

**Schedule**: 20,000 iterations (~many epochs depending on dataset size) with validation every 2,000 iterations.  
**Augmentations**: Random resize, horizontal/vertical flips, photo-metric distortion.  
**Monitoring**: We'll set up TensorBoard to view loss curves and sample predictions.

In [ ]:
# Load TensorBoard for live monitoring
%load_ext tensorboard
%tensorboard --logdir {WORK_DIR} --port 6006

In [ ]:
%%time
!cd /content/ncct-segmentation && python tools/train.py {UPDATED_CFG_PATH} --work-dir {WORK_DIR} --amp

### Training Results

After training completes, checkpoints (.pth files) and logs are saved in the work directory:

In [ ]:
import glob

# List checkpoint files
ckpts = sorted(glob.glob(os.path.join(WORK_DIR, "*.pth")))
print(f"Checkpoints found: {len(ckpts)}")
for ckpt in ckpts:
    size_mb = os.path.getsize(ckpt) / (1024 * 1024)
    print(f"  {os.path.basename(ckpt):30s} {size_mb:.1f} MB")

# Show latest training log
log_files = sorted(glob.glob(os.path.join(WORK_DIR, "*.log*")))
if log_files:
    print(f"\nLog file: {os.path.basename(log_files[-1])}")

### Plot Loss and Metric Curves

Parse the training log and visualize the learning curves.

In [ ]:
import re
import json

def parse_log(json_log_path):
    """Parse MMSegmentation JSON log file."""
    iters = []
    losses = []
    dices = []
    ious = []
    val_iters = []

    with open(json_log_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue

            iteration = entry.get("iteration")
            if entry.get("mode") == "train" and "loss" in entry:
                iters.append(iteration)
                losses.append(entry["loss"])
            elif entry.get("mode") == "val" and "mDice" in entry:
                val_iters.append(iteration)
                dices.append(entry.get("mDice", 0))
                ious.append(entry.get("mIoU", 0))

    return iters, losses, val_iters, dices, ious

log_paths = sorted(glob.glob(os.path.join(WORK_DIR, "*.log.json")))
if log_paths:
    iters, losses, val_iters, dices, ious = parse_log(log_paths[-1])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Loss curve
    if iters and losses:
        axes[0].plot(iters, losses, "b-", alpha=0.7, label="Train Loss")
        axes[0].set_xlabel("Iteration")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Training Loss")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

    # Validation metrics
    if val_iters and dices:
        axes[1].plot(val_iters, dices, "g-o", alpha=0.7, label="mDice")
        if ious:
            axes[1].plot(val_iters, ious, "r-s", alpha=0.7, label="mIoU")
        axes[1].set_xlabel("Iteration")
        axes[1].set_ylabel("Score")
        axes[1].set_title("Validation Metrics")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    if dices:
        print(f"Best val mDice: {max(dices):.4f} at iter {val_iters[dices.index(max(dices))]}")
    if ious:
        print(f"Best val mIoU:  {max(ious):.4f} at iter {val_iters[ious.index(max(ious))]}")
else:
    print("No log files found yet. Training may still be running or failed.")

---
## 5. Evaluation

Run the best checkpoint on the test set.

In [ ]:
# Find the best checkpoint (latest or highest Dice)
# In MMSegmentation, the latest checkpoint is typically at max iterations
ckpts = sorted(glob.glob(os.path.join(WORK_DIR, "iter_*.pth")))
if not ckpts:
    # Fallback: any .pth
    ckpts = sorted(glob.glob(os.path.join(WORK_DIR, "*.pth")))

if ckpts:
    BEST_CKPT = ckpts[-1]  # Last checkpoint = most trained
    print(f"Using checkpoint: {BEST_CKPT}")
else:
    print("No checkpoints found!")

In [ ]:
%%time
if ckpts:
    !cd /content/ncct-segmentation && python tools/test.py \
        {UPDATED_CFG_PATH} \
        {BEST_CKPT} \
        --show-dir {WORK_DIR}/preds \
        --out {WORK_DIR}/results.pkl

### Test Metrics Summary

Let's parse the test output to see detailed per-class metrics.

In [ ]:
import subprocess

# Re-run test with more verbose output captured
result = subprocess.run(
    ["python", "tools/test.py", UPDATED_CFG_PATH, BEST_CKPT, "--out", f"{WORK_DIR}/results.pkl"],
    capture_output=True, text=True, cwd="/content/ncct-segmentation"
)

# Extract the metric summary lines
for line in result.stdout.split("\n"):
    if any(kw in line for kw in ["Dice", "IoU", "mDice", "mIoU", "aAcc"]):
        print(line)
    if "OrderedDict" in line and ("Dice" in line or "IoU" in line):
        print(line)

# If stdout was empty, try stderr (mmseg logs metrics to stderr in some versions)
if not any(kw in result.stdout for kw in ["Dice", "mDice"]):
    for line in result.stderr.split("\n"):
        if any(kw in line for kw in ["Dice", "IoU", "mDice", "mIoU", "aAcc"]):
            print(line)

### Visualize Predictions

Compare ground truth masks with model predictions on test samples.

In [ ]:
from mmengine.config import Config
from mmseg.apis import init_model, inference_model, show_result_pyplot

# Load the trained model
model = init_model(UPDATED_CFG_PATH, BEST_CKPT, device="cuda:0")

# Get test images
test_img_dir = os.path.join(cfg.data_root, "test/images")
test_mask_dir = os.path.join(cfg.data_root, "test/masks")
test_images = sorted(os.listdir(test_img_dir))[:6]  # Show 6 test samples

fig, axes = plt.subplots(len(test_images), 4, figsize=(16, 4 * len(test_images)))

for i, fname in enumerate(test_images):
    img_path = os.path.join(test_img_dir, fname)
    mask_path = os.path.join(test_mask_dir, fname)

    # Load ground truth
    img_np = np.array(Image.open(img_path).convert("L"))
    mask_np = np.array(Image.open(mask_path).convert("L"))
    mask_bin = (mask_np > 127).astype(np.uint8)

    # Run inference
    result = inference_model(model, img_path)
    pred = result.pred_sem_seq.data.cpu().numpy()  # shape: (H, W), values: 0 or 1

    # Plot
    axes[i][0].imshow(img_np, cmap="gray")
    axes[i][0].set_title("Input NCCT")
    axes[i][0].axis("off")

    axes[i][1].imshow(mask_bin, cmap="gray")
    axes[i][1].set_title("Ground Truth")
    axes[i][1].axis("off")

    axes[i][2].imshow(pred, cmap="gray")
    axes[i][2].set_title("Prediction")
    axes[i][2].axis("off")

    # Overlay: prediction outline on input
    from scipy.ndimage import binary_dilation
    outline = binary_dilation(pred, iterations=1) ^ pred  # edge mask
    overlay = np.stack([img_np] * 3, axis=-1).astype(np.float32)
    overlay[:, :, 0] = np.where(outline, 255, overlay[:, :, 0])  # red outline
    overlay[:, :, 1] = np.where(outline, 0, overlay[:, :, 1])
    overlay[:, :, 2] = np.where(outline, 0, overlay[:, :, 2])
    axes[i][3].imshow(overlay.astype(np.uint8))
    axes[i][3].set_title("Overlay (Pred edge)")
    axes[i][3].axis("off")

plt.tight_layout()
plt.show()

---
## 6. Save and Export

Save the trained weights and optionally mount Google Drive for persistent storage.

In [ ]:
from google.colab import drive

# Mount Google Drive
DRIVE_MOUNT = "/content/drive"
drive.mount(DRIVE_MOUNT)

# Copy best checkpoint to Drive
import shutil
DRIVE_DST = "/content/drive/MyDrive/ncct_segmentation_results"
os.makedirs(DRIVE_DST, exist_ok=True)

# Copy checkpoint
ckpt_name = os.path.basename(BEST_CKPT)
shutil.copy2(BEST_CKPT, os.path.join(DRIVE_DST, ckpt_name))
print(f"Copied {ckpt_name} to Drive")

# Copy config
shutil.copy2(UPDATED_CFG_PATH, os.path.join(DRIVE_DST, "unet_stroke_ncct_colab.py"))
print("Copied config to Drive")

# Copy results
if os.path.exists(f"{WORK_DIR}/results.pkl"):
    shutil.copy2(f"{WORK_DIR}/results.pkl", os.path.join(DRIVE_DST, "results.pkl"))
    print("Copied results to Drive")

print(f"\nFiles saved to: {DRIVE_DST}")

---
## 7. Try Other Architectures (Optional)

MMSegmentation supports many architectures beyond U-Net. Below are configs you can try for potentially better accuracy:

| Model | Config Path | Notes |
|---|---|---|
| U-Net (current) | `configs/ncct/unet_stroke_ncct.py` | Baseline |
| PSPNet | `configs/pspnet/pspnet_r50-d8_4xb2-40k_cityscapes-512x1024.py` | Good global context |
| DeepLabV3+ | `configs/deeplabv3plus/deeplabv3plus_r50-d8_4xb2-40k_cityscapes-512x1024.py` | Strong general architecture |
| DeepLabV3 | `configs/deeplabv3/deeplabv3_r50-d8_4xb2-40k_cityscapes-512x1024.py` | Atrous convolution |
| UNet-Swin | `configs/swin/upernet_swin_tiny_patch4_window7_512x512_160k_ade20k.py` | Transformer backbone |

To try a different architecture, create a new config in `configs/ncct/` that:
1. Inherits from the desired base model
2. Sets `num_classes=2`
3. Uses `StrokeNCCTDataset`
4. Uses the same dataloader/pipeline settings

In [ ]:
# Example: Create a DeepLabV3+ config for NCCT
deeplab_cfg = """
_base_ = [
    '../_base_/models/deeplabv3plus_r50-d8.py',
    '../_base_/default_runtime.py',
    '../_base_/schedules/schedule_40k.py'
]

model = dict(
    data_preprocessor=dict(
        type='SegDataPreProcessor',
        mean=[0, 0, 0],
        std=[255, 255, 255],
        bgr_to_rgb=False),
    decode_head=dict(num_classes=2),
    auxiliary_head=dict(num_classes=2))

dataset_type = 'StrokeNCCTDataset'
data_root = '/content/data/ncct/'

train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='RandomResize', scale=(256, 256), keep_ratio=True),
    dict(type='RandomFlip', prob=0.5, direction='horizontal'),
    dict(type='RandomFlip', prob=0.5, direction='vertical'),
    dict(type='PackSegInputs'),
]

test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs'),
]

train_dataloader = dict(
    batch_size=8, num_workers=2, persistent_workers=True,
    sampler=dict(type='InfiniteSampler', shuffle=True),
    dataset=dict(type=dataset_type, data_root=data_root,
                 data_prefix=dict(img_path='train/images', seg_map_path='train/masks'),
                 pipeline=train_pipeline))

val_dataloader = dict(
    batch_size=1, num_workers=2, persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(type=dataset_type, data_root=data_root,
                 data_prefix=dict(img_path='val/images', seg_map_path='val/masks'),
                 pipeline=test_pipeline))

test_dataloader = val_dataloader

val_evaluator = dict(type='IoUMetric', iou_metrics=['mDice', 'mIoU'])
test_evaluator = val_evaluator
"""

DEEPLAB_CFG_PATH = "/content/ncct-segmentation/configs/ncct/deeplabv3plus_stroke_ncct.py"
with open(DEEPLAB_CFG_PATH, "w") as f:
    f.write(deeplab_cfg)
print(f"Created {DEEPLAB_CFG_PATH}")

In [ ]:
# Train DeepLabV3+ (uncomment to run)
# DEEPLAB_WORK_DIR = "/content/work_dirs/deeplabv3p_stroke_ncct"
# !cd /content/ncct-segmentation && python tools/train.py {DEEPLAB_CFG_PATH} --work-dir {DEEPLAB_WORK_DIR} --amp

---
## Summary

1. ✅ **Data prepared**: NCCT dataset organized in MMSegmentation format
2. ✅ **Config optimized**: Normalization for medical images, class-weighted loss, proper augmentation
3. ✅ **Model trained**: U-Net with 20K iterations + AMP
4. ✅ **Results saved**: Checkpoints, logs, and predictions

**Next steps to improve accuracy:**
- Train for longer (40K+ iterations)
- Try DeepLabV3+, PSPNet, or Swin-UNet architectures
- Use cross-validation
- Fine-tune data augmentation parameters
- Collect more labeled training data